In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
from pulp import LpProblem, LpMinimize

np.random.seed(42)

In [33]:
df_demand = pd.read_excel("data/demand.xlsx")
print(df_demand.head())

  Country_Code   Unit Demand
0           AR  9.545572e+05
1           AT  3.370552e+06
2           AU  3.321179e+06
3           BE  4.993605e+06
4           BR  2.478707e+06


In [41]:
df_distribution = pd.read_excel("data/distribution.xlsx")
print(df_distribution.head())

  Site Code PRD Site Code GDC  Unit Distribution  Transfer cost per unit
0          PLLU          USNE           10949933                0.076424
1          PLLU          USSA                  0                0.313274
2          PLLU          PLWA            8411057                0.430141
3          PLLU          ESMA            2457791                0.149379
4          PLLU          ARBU                  0                0.449957


In [35]:
df_location = pd.read_excel("data/locations.xlsx")
print(df_location.head())

  Type        Country Country_code Site_code       City      X_cord     Y_cord
0  PMP         Poland           PL      PLLU     Lublin   22.580737  51.276011
1  PMP  United States           US      USAM   Amarillo -102.851814  32.957891
2  PMP    South Korea           KR      KRCH     Chinju  128.107621  35.179982
3  PMP         Turkey           TR      TRME  Merkezköy   40.009251  40.832193
4  GDC            USA           US      USNE   New York  -73.935242  40.730610


In [36]:
df_prices = pd.read_excel("data/prices.xlsx")
print(df_prices.head())

  Country_Code  Price per Unit in USD
0           AR                   3.53
1           AT                   4.43
2           AU                   4.25
3           BE                   4.32
4           BR                   4.23


In [37]:
df_production = pd.read_excel("data/production.xlsx")
print(df_production.head())

         Country Country_code Site Code  Unit Production
0         Poland           PL      PLLU         37298613
1  United States           US      USAM         41134963
2    South Korea           KR      KRCH         24640961
3         Turkey           TR      TRME         40168561


In [42]:
df_distribution.columns = [c.strip() for c in df_distribution.columns]
df_location.columns     = [c.strip() for c in df_location.columns]
df_production.columns   = [c.strip() for c in df_production.columns]
df_demand.columns       = [c.strip() for c in df_demand.columns]

## Problem statement

In [ ]:
# cost minimization

model = LpProblem("CostMinimization", LpMinimize)


In [22]:
# production
p = pulp.LpVariable.dicts(
    "production",
    (row["Site Code"] for index, row in df_production.iterrows()),
    lowBound=0,
    cat='Integer'
)

# prices - fixed

# demand - fixed

# transfer

t = pulp.LpVariable.dicts(
    "transfer",
    ((row["Site Code"]) for index, row in df_production.iterrows()),
    lowBound=0,
    cat='Integer'
)



{'PLLU': P_PLLU, 'USAM': P_USAM, 'KRCH': P_KRCH, 'TRME': P_TRME}

## Task 1

In [45]:
supply_cost = np.sum(df_distribution["Transfer cost per unit"] * df_distribution["Unit Distribution"])
print(f"Cost of Supply from Manufacturing to Distribution: {supply_cost:.2f}")

Cost of Supply from Manufacturing to Distribution: 58714245.92


## Task 2

In [48]:
df_location

,Type,Country,Country_code,Site_code,City,X_cord,Y_cord
0,PMP,Poland,PL,PLLU,Lublin,22.580737,51.276011
1,PMP,United States,US,USAM,Amarillo,-102.851814,32.957891
2,PMP,South Korea,KR,KRCH,Chinju,128.107621,35.179982
3,PMP,Turkey,TR,TRME,Merkezköy,40.009251,40.832193
4,GDC,USA,US,USNE,New York,-73.935242,40.730610
...,...,...,...,...,...,...,...
96,MKT,Turkey,TR,AR,Ankara,38.963700,35.243300
97,MKT,Ukraine,UA,AR,Kyiv,48.379400,31.165600
98,MKT,United States of America,US,AR,Washington,37.090200,-95.712900
99,MKT,Vietnam,VN,AR,Hanoi,14.058300,108.277200


In [47]:
df_distribution

,Site Code PRD,Site Code GDC,Unit Distribution,Transfer cost per unit
0,PLLU,USNE,10949933,0.076424
1,PLLU,USSA,0,0.313274
2,PLLU,PLWA,8411057,0.430141
3,PLLU,ESMA,2457791,0.149379
4,PLLU,ARBU,0,0.449957
5,PLLU,CNBE,4131069,0.558353
6,PLLU,JPTO,0,0.329165
7,PLLU,INMU,11348763,0.113696
8,USAM,USNE,3125827,0.228482
9,USAM,USSA,432765,0.589694


In [63]:
df_dist1 = pd.merge(df_distribution, df_location[["Site_code", "X_cord", "Y_cord"]], left_on="Site Code PRD", right_on="Site_code", how="left")
df_dist2 = pd.merge(df_dist1, df_location[["Site_code", "X_cord", "Y_cord"]], left_on="Site Code GDC", right_on="Site_code", how="left")

import math
def haversine(lon1, lat1, lon2, lat2):
    # returns distance in km
    R = 6371.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))

df_dist2["Distance"] = df_dist2.apply(lambda x: haversine(x["X_cord_x"], x["Y_cord_x"], x["X_cord_y"], x["Y_cord_y"]), axis=1)

cost = np.sum(0.1 / 1000 * df_dist2["Distance"] * df_dist2["Unit Distribution"])
print(cost)

77739592.72689913
